In [8]:
import gmsh

gmsh.initialize()


gmsh.model.add("glacier_terminus")

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Ly = 750.0
Lz = 125.0

# -------------------------------------------------------------------------
# Crevasse dimensions [m]
# -------------------------------------------------------------------------

lx = 5.0
ly = 10.0
lz = 10.0

# Offset of each crevasse from the centerline
x_offset = 100.0

# -------------------------------------------------------------------------
# Mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 1.0
h_max = 19.9

gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)
gmsh.option.setNumber("Mesh.MeshSizeFactor", 1.0)
# -------------------------------------------------------------------------
# Glacier geometry
# -------------------------------------------------------------------------

glacier = gmsh.model.occ.addBox(
    0.0,
    0.0,
    0.0,
    Lx,
    Ly,
    Lz,
)

# -------------------------------------------------------------------------
# Two opposite crevasses
#
# Crevasse 1:
#   starts from y = 0 and extends into the domain in +y
#
# Crevasse 2:
#   starts from y = Ly and extends into the domain in -y
#
# Their x locations are symmetrically offset from x = Lx / 2
# -------------------------------------------------------------------------

# Left-offset crevasse
crevasse_1_x0 = 0.5 * Lx - x_offset - 0.5 * lx
crevasse_1_y0 = 0.0
crevasse_1_z0 = Lz - lz

crevasse_1 = gmsh.model.occ.addBox(
    crevasse_1_x0,
    crevasse_1_y0,
    crevasse_1_z0,
    lx,
    ly,
    lz,
)

# Right-offset crevasse, entering from the opposite y boundary
crevasse_2_x0 = 0.5 * Lx + x_offset - 0.5 * lx
crevasse_2_y0 = Ly - ly
crevasse_2_z0 = Lz - lz

crevasse_2 = gmsh.model.occ.addBox(
    crevasse_2_x0,
    crevasse_2_y0,
    crevasse_2_z0,
    lx,
    ly,
    lz,
)

# -------------------------------------------------------------------------
# Subtract both crevasses from the glacier
# -------------------------------------------------------------------------

domain, _ = gmsh.model.occ.cut(
    [(3, glacier)],
    [(3, crevasse_1), (3, crevasse_2)],
    removeObject=True,
    removeTool=True,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Physical volume
# -------------------------------------------------------------------------

volume_tags = [tag for dim, tag in domain if dim == 3]

gmsh.model.addPhysicalGroup(3, volume_tags, 1)
gmsh.model.setPhysicalName(3, 1, "GLACIER")

# -------------------------------------------------------------------------
# Generate tetrahedral mesh
# -------------------------------------------------------------------------

gmsh.model.mesh.generate(3)

gmsh.write("glacier_{:.0f}.msh".format(x_offset))

gmsh.finalize()

Info    : Meshing 1D...                                                                                      
Info    : [  0%] Meshing curve 13 (Line)
Info    : [ 10%] Meshing curve 14 (Line)
Info    : [ 10%] Meshing curve 15 (Line)
Info    : [ 10%] Meshing curve 16 (Line)
Info    : [ 20%] Meshing curve 17 (Line)
Info    : [ 20%] Meshing curve 18 (Line)
Info    : [ 20%] Meshing curve 19 (Line)
Info    : [ 20%] Meshing curve 20 (Line)
Info    : [ 30%] Meshing curve 21 (Line)
Info    : [ 30%] Meshing curve 23 (Line)
Info    : [ 30%] Meshing curve 24 (Line)
Info    : [ 40%] Meshing curve 25 (Line)
Info    : [ 40%] Meshing curve 26 (Line)
Info    : [ 40%] Meshing curve 27 (Line)
Info    : [ 40%] Meshing curve 28 (Line)
Info    : [ 50%] Meshing curve 29 (Line)
Info    : [ 50%] Meshing curve 30 (Line)
Info    : [ 50%] Meshing curve 31 (Line)
Info    : [ 60%] Meshing curve 32 (Line)
Info    : [ 60%] Meshing curve 33 (Line)
Info    : [ 60%] Meshing curve 34 (Line)
Info    : [ 60%] Meshing curv

In [1]:
import meshio
import os

os.makedirs("adaptive", exist_ok=True)
s = [15,25,50,100]
for size in s:
    mesh = meshio.read(f"glacier_{size}.msh")

    cells = mesh.get_cells_type("tetra")
    points = mesh.points

    meshio.write(f"adaptive/mesh_{size}.xdmf", meshio.Mesh(
        points=points,
        cells={"tetra": cells}))